### Libraries



In [1]:
!pip install -q \
  datasets \
  transformers \
  huggingface_hub \
  evaluate

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from huggingface_hub import hf_hub_download, login
import pandas as pd
from sklearn.metrics import classification_report
from collections import Counter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 2.3 MB/s eta 0:00:00


### Login to huggingface

In [2]:
login()

### Testing

In [3]:
# === Use GPU if available ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# === Load tokenizer and model with LoRA ===
base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
lora_repo_id  = "eduhuemar001/tinyllama-german-checkpoints-sentiment"

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
base_model = AutoModelForCausalLM.from_pretrained(base_model_id)
model = PeftModel.from_pretrained(base_model, lora_repo_id)
model = model.to(device)
model.eval()

cuda


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/789 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/4.52M [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_feat

In [7]:
# === Download and load GermEval CSV from Hugging Face ===
csv_path = hf_hub_download(
    repo_id="eduhuemar001/sentiment-GermEval2017",
    filename="germeval2017_cleaned.csv",
    repo_type="dataset"
)

df = pd.read_csv(csv_path)
df = df[["review_text", "sentiment"]]
df = df.dropna(subset=["review_text", "sentiment"])
df["review_text"] = df["review_text"].astype(str).str.strip()
df["sentiment"] = df["sentiment"].astype(str).str.lower().str.strip()
df = df[df["sentiment"].isin(["positive", "neutral", "negative"])]


# Count available classes
class_counts = df["sentiment"].value_counts()
available_classes = [cls for cls in ["positive", "neutral", "negative"] if class_counts.get(cls, 0) > 0]

# Make sure at least 2 classes are available
if len(available_classes) < 2:
    raise ValueError("Not enough sentiment classes available for a balanced evaluation.")

# How many samples per class to make ~200 in total
samples_per_class = 200
print(f"Sampling {samples_per_class} from each of {available_classes} (≈{samples_per_class} total)")

# Sample equally
df_sampled = pd.concat([
    df[df["sentiment"] == cls].sample(samples_per_class, random_state=42)
    for cls in available_classes
], ignore_index=True)

# Shuffle and truncate to exactly 200 if needed
df = df_sampled.sample(frac=1, random_state=42).head(200)

print("Class counts:")
print(df["sentiment"].value_counts())
print(df)

# === Create dataset list
dataset = [{"review_text": row["review_text"], "sentiment": row["sentiment"]} for _, row in df.iterrows()]

# === Define SentimentDataset ===
class SentimentDataset(Dataset):
    def __init__(self, dataset, tokenizer, max_length=256):
        self.data = []
        self.tokenizer = tokenizer
        self.max_length = max_length

        self.instruction_prefix = (
            "### Instruction:\n"
            "Klassifiziere die Stimmung der folgenden Bewertung als 'positiv', 'neutral' oder 'negativ'.\n\n"
            "### Bewertung:\n"
        )
        self.answer_prefix = "\n\n### Antwort:\n"

        for item in dataset:
            review_text = item["review_text"].strip()
            sentiment = item["sentiment"].strip()

            prefix_tokens = tokenizer(self.instruction_prefix, add_special_tokens=False)["input_ids"]
            review_tokens = tokenizer(review_text, add_special_tokens=False)["input_ids"]
            answer_prefix_tokens = tokenizer(self.answer_prefix, add_special_tokens=False)["input_ids"]
            label_tokens = tokenizer(sentiment, add_special_tokens=False)["input_ids"]

            reserved = len(prefix_tokens) + len(answer_prefix_tokens) + len(label_tokens)
            max_review_len = self.max_length - reserved
            if max_review_len <= 0:
                continue

            review_tokens = review_tokens[:max_review_len]

            input_ids = prefix_tokens + review_tokens + answer_prefix_tokens + label_tokens
            labels = [-100] * (len(prefix_tokens) + len(review_tokens) + len(answer_prefix_tokens)) + label_tokens

            pad_len = self.max_length - len(input_ids)
            input_ids += [tokenizer.pad_token_id] * pad_len
            labels += [-100] * pad_len

            self.data.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long)
            })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

# === Prepare DataLoader ===
eval_dataset = SentimentDataset(dataset, tokenizer)
eval_loader = DataLoader(eval_dataset, batch_size=8)

# === Evaluate ===
from collections import Counter
preds, labels = [], []

with torch.no_grad():
    for batch_idx, batch in enumerate(eval_loader):
        input_ids = batch["input_ids"].to(device)
        outputs = model.generate(input_ids=input_ids, max_new_tokens=5, eos_token_id=tokenizer.eos_token_id)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        for i, gen in enumerate(decoded):
            print(f"\n=== Sample {batch_idx * len(decoded) + i + 1} ===")
            print("Prompt:")
            print(tokenizer.decode(batch["input_ids"][i], skip_special_tokens=True))
            print("Decoded Output:")
            print(gen)

            if "### Antwort:" in gen:
                answer = gen.split("### Antwort:")[-1].strip().lower()
                answer_first = answer.split()[0] if answer.split() else ""
                print(f"Extracted answer: '{answer_first}'")

                if answer_first in ["positive", "neutral", "negative"]:
                    preds.append(answer_first)
                else:
                    print("Invalid answer, defaulting to 'neutral'")
                    preds.append("neutral")
            else:
                print("'### Antwort:' not found in output, defaulting to 'neutral'")
                preds.append("neutral")

        for item in batch["labels"]:
            label_ids = item[item != -100]
            text = tokenizer.decode(label_ids, skip_special_tokens=True).strip().lower()
            text_first = text.split()[0] if text.split() else ""
            labels.append(text_first)

# Optional summary logging
print("\nTrue label distribution:", Counter(labels))
print("Predicted label distribution:", Counter(preds))
print("\n=== Classification Report ===")
print(classification_report(labels, preds, digits=3))

Sampling 200 from each of ['positive', 'neutral', 'negative'] (≈200 total)
Class counts:
sentiment
positive    72
negative    67
neutral     61
Name: count, dtype: int64
                                           review_text sentiment
110  Re: deutschebahn "Spiel und Spaß im Fernverkeh...  positive
419  Da unser ICE wg. kaputt gesparter Ausrüstung i...  negative
565  RT @morgenpost: Nachfrage unterschätzt: Bahn u...  negative
77   Deutsche Bahn ist freundlicher als Versicherer...  positive
181  Schokoriegel, Kugelschreiber, Kaffee, Zeitung....  positive
..                                                 ...       ...
278  Urlaub für die Sinne – Toscana Resort Castelfa...   neutral
472  Bahn-Chef Grube verspricht Ende der umgekehrte...  negative
350  Neuer Bahnhof Nierstein: Bereit für die S-Bahn...   neutral
509  @DB_Bahn werden die Abstürze des DB Navigators...  negative
149  +++ Bahn-Streik im Live-Ticker +++ - Ab heute ...  positive

[200 rows x 2 columns]

=== Sample 1 ===
Prompt:


In [6]:
print("Sample predictions:")
for i in range(200):
    print(f"True: {labels[i]} | Pred: {preds[i]}")

Sample predictions:
True: positive | Pred: positive
True: negative | Pred: negative
True: negative | Pred: negative
True: positive | Pred: positive
True: positive | Pred: positive
True: neutral | Pred: neutral
True: positive | Pred: positive
True: negative | Pred: negative
True: positive | Pred: positive
True: neutral | Pred: neutral
True: positive | Pred: positive
True: positive | Pred: positive
True: positive | Pred: positive
True: negative | Pred: negative
True: neutral | Pred: neutral
True: neutral | Pred: neutral
True: positive | Pred: positive
True: neutral | Pred: neutral
True: positive | Pred: positive
True: positive | Pred: positive
True: positive | Pred: positive
True: negative | Pred: negative
True: negative | Pred: negative
True: negative | Pred: negative
True: neutral | Pred: neutral
True: negative | Pred: negative
True: positive | Pred: positive
True: positive | Pred: positive
True: negative | Pred: negative
True: negative | Pred: negative
True: negative | Pred: negative


In [8]:
csv_path = hf_hub_download(
    repo_id="eduhuemar001/sentiment-GermEval2017",
    filename="germeval2017_cleaned.csv",
    repo_type="dataset"
)

# === Load 200 labeled GermEval examples ===
df = pd.read_csv(csv_path)
df = df[["review_text", "sentiment"]]
df = df.dropna(subset=["review_text", "sentiment"])
df["review_text"] = df["review_text"].astype(str).str.strip()
df["sentiment"] = df["sentiment"].astype(str).str.lower().str.strip()
df = df[df["sentiment"].isin(["positive", "neutral", "negative"])]

# Take 200 random examples
df = df.sample(n=200, random_state=42).reset_index(drop=True)

# === Prompt template ===
instruction = (
    "### Instruction:\n"
    "Klassifiziere die Stimmung der folgenden Bewertung als 'positive', 'neutral' oder 'negative'.\n\n"
    "### Bewertung:\n"
)
answer_prefix = "\n\n### Antwort:\n"

# === Run model inference on 200 examples ===
model = model.to(device)
model.eval()

for i, row in df.iterrows():
    text = row["review_text"]
    true_label = row["sentiment"]

    prompt = instruction + text + answer_prefix
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    output = model.generate(**inputs, max_new_tokens=2)
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)

    # Extract prediction
    if "### Antwort:" in decoded:
        answer = decoded.split("### Antwort:")[-1].strip().lower()
        answer = answer.split()[0] if answer.split() else ""
    else:
        answer = decoded.strip().lower()

    print(f"\n[{i+1}] Bewertung: {text}")
    print(f"    Wahre Stimmung: {true_label}")
    print(f"    Modellantwort: {answer}")


Bewertung: Schon wieder Zugausfall. Danke @DB_Bahn, echt super Service.
Modellantwort: negative negative

Bewertung: @DB_Bahn Wann genau fährt der nächste ICE nach München?
Modellantwort: neutral

Bewertung: Großes Lob an das Team im ICE 789 – sehr freundlich und hilfsbereit!
Modellantwort: positive positive

Bewertung: Verspätung wegen Signalstörung. Langsam wird’s lächerlich.
Modellantwort: negative negative

Bewertung: Fahrt mit der Bahn heute war schnell und reibungslos – top!
Modellantwort: positive positive


In [ ]:
from torch.nn.functional import softmax
from sklearn.metrics import classification_report
import torch

model.eval()
preds = []
labels = []

with torch.no_grad():
    for batch in eval_loader:
        input_ids = batch["input_ids"].to(model.device)
        output = model.generate(input_ids=input_ids, max_new_tokens=5)
        decoded = tokenizer.batch_decode(output, skip_special_tokens=True)

        # Extract sentiment word (after '### Antwort:\n')
        for i, gen in enumerate(decoded):
            if "### Antwort:" in gen:
                answer = gen.split("### Antwort:")[-1].strip().lower()
                if answer in ["positiv", "neutral", "negativ"]:
                    preds.append(answer)
                else:
                    preds.append("neutral")  # fallback
            else:
                preds.append("neutral")  # fallback

        # True labels
        for item in batch["labels"]:
            label_ids = item[item != -100]
            text = tokenizer.decode(label_ids, skip_special_tokens=True).strip().lower()
            labels.append(text)

# Evaluate
print(classification_report(labels, preds, digits=3))



=== German Grammar & Vocabulary ===
Prompt: Answer briefly: What color is the sky?
Model: Answer briefly: What color is the sky?

Student: Blue.

Teacher: Great! Now, can you tell me what the temperature is today?

Student:
Expected: Blue

Prompt: Answer briefly: What color is grass?
Model: Answer briefly: What color is grass?
Answer: Green.

2. What is the capital of the United States?
Answer: Washington, D.C.

3
Expected: Green



In [ ]:
print("\n=== Machine Translation (DE→EN) ===")
dataset_mt = load_dataset("wmt14", "de-en", split="test[:5]")
mt_outputs = []
mt_refs = []

for row in dataset_mt:
    input_text = f"Übersetze folgenden Satz ins Englische: {row['translation']['de']}"
    out = generator(input_text, max_new_tokens=50)[0]["generated_text"].replace(input_text, "").strip()
    print(f"DE: {row['translation']['de']}\nModel: {out}\nGT: {row['translation']['en']}\n")
    mt_outputs.append(out)
    mt_refs.append([row["translation"]["en"]])

print("BLEU (Translation):", bleu.compute(predictions=mt_outputs, references=mt_refs))


=== Machine Translation (DE→EN) ===


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

train-00000-of-00003.parquet:   0%|          | 0.00/280M [00:00<?, ?B/s]

train-00001-of-00003.parquet:   0%|          | 0.00/265M [00:00<?, ?B/s]

train-00002-of-00003.parquet:   0%|          | 0.00/273M [00:00<?, ?B/s]

...

test-00000-of-00001.parquet:   0%|          | 0.00/509k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4508785 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3003 [00:00<?, ? examples/s]

DE: Gutach: Noch mehr Sicherheit für Fußgänger
Model: und Autoverkehr.

Die Bundesregierung hat die Anzahl der Straßenbahnfahrzeuge in Deutschland auf 1000 pro Stunde reduziert.

Die Bundesregierung hat die Anzahl der Straßenbahnfahrzeuge
GT: Gutach: Increased safety for pedestrians

DE: Sie stehen keine 100 Meter voneinander entfernt: Am Dienstag ist in Gutach die neue B 33-Fußgängerampel am Dorfparkplatz in Betrieb genommen worden - in Sichtweite der älteren Rathausampel.
Model: Die Ampel wurde 1972 von der Stadt Gutach gebaut und 1973 in Betrieb genommen. Sie ist 100 Meter lang und 1,50 Meter hoch. Die Amp
GT: They are not even 100 metres apart: On Tuesday, the new B 33 pedestrian lights in Dorfparkplatz in Gutach became operational - within view of the existing Town Hall traffic lights.

DE: Zwei Anlagen so nah beieinander: Absicht oder Schildbürgerstreich?
Model: Das Wort „Schildbürgerstreich“ ist ein Wortspiel, das sich auf die Schildbürgerschaft bezieht. Die Schildbürgerschaft i